In [1]:
import warnings
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

/mnt/c/Users/wstre/projects/lca-lc-foundations/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [5]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [6]:
from langchain.agents import create_agent

from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="openrouter/owl-alpha",
    temperature=0.5,
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [7]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='9824ac26-4f29-444c-86fa-8e7417984cab'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openrouter/owl-alpha', 'id': 'gen-1779047326-9QAwHA8Ke8Xya2TWhmQ2', 'created': 1779047326, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019e377b-dfc9-7180-8b9c-8b764c8d2949-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '911850ae-24ec-4e57-9296-96d4763a1ac4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 367, 'output_tokens': 21, 'total_tokens': 388, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "query": "langchain-mcp-adapters library",\n  

In [9]:
print(response['messages'][-1].content)

The `langchain-mcp-adapters` library is a tool designed to bridge the gap between **LangChain/LangGraph** and Anthropic's **Model Context Protocol (MCP)**. It allows developers to easily integrate external tools and data sources into their AI agents.

### Key Features:
*   **Seamless Integration:** It converts MCP tools into LangChain- and LangGraph-compatible tools, allowing you to tap into the growing ecosystem of MCP servers without writing custom adapters.
*   **Multi-Server Support:** The library enables agents to pull from multiple MCP servers simultaneously, making it easier to combine different tools for more powerful applications.
*   **Flexible Connections:** It supports connecting to MCP servers via `stdio` (local) or `Streamable HTTP` (remote), with automatic fallback to SSE for compatibility with legacy implementations.
*   **Authentication:** It supports custom headers in SSE connections for secure authentication.

### Why use it?
MCP is becoming a standard for connecting

## Online MCP

In [10]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [11]:
agent = create_agent(
    model=model,
    tools=tools,
)

In [15]:
question = HumanMessage(content="What time is it in Birmingham, AL?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='What time is it in Birmingham, AL?', additional_kwargs={}, response_metadata={}, id='0004d104-1a5e-4bbf-b44e-73627448e556'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openrouter/owl-alpha', 'id': 'gen-1779047602-zszcV2rSdFF7a3rQWnnv', 'created': 1779047602, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019e3780-189e-7c31-a82a-40f34c4e4f13-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/Chicago'}, 'id': 'd6954c1f-3f68-4630-b557-5dc382c0ee74', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 457, 'output_tokens': 18, 'total_tokens': 475, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "timezone": "America/Chicago",\n  "datetime": "2026-05-17T14:53:28